Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2025/26 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# OLS-Based, Univariate Linear Regression

In [ ]:
import numpy as np
from numpy.linalg import matrix_rank, norm
from scipy.linalg import svd, diagsvd, inv, pinv
import matplotlib.pyplot as plt

## Linear Univariate Model

In [ ]:
# set up a ground truth model
x = np.arange(4)
m_truth = 3  # slope
n_truth = 2  # intercept
X = np.array([np.ones_like(x), x]).T  # tall/thin, full column rank X
theta_truth = np.array([[m_truth], [n_truth]])
y_truth = X @ theta_truth # get ground truth data
y_truth, norm(y_truth)**2

In [ ]:
R = matrix_rank(X)
[M, N] = X.shape
# SVD of model matrix
[U, s, Vt] = svd(X)
V = Vt.T
S = diagsvd(s, M, N)
# SVD-based U-projections matrices
P_CS = U[:,:R] @ U[:,:R].T  # column space
P_LNS = U[:,R:] @ U[:,R:].T  # left null space

In [ ]:
# find model parameters with all the data == overfitting the model

# for that we need the inverse of X
# inverse (here left inverse) via pinv:
X_li = pinv(X)
# general concept of the pseudo inverse:
X_pseudo_inv_svd_general = V @ diagsvd(1/s, N, M) @ U.T
# formulation for the left inverse using the S matrix:
X_li_svd = V @ (inv(S.T @ S) @ S.T) @ U.T
X_li, np.allclose(X_li, X_pseudo_inv_svd_general), np.allclose(X_li, X_li_svd)


## Case 1: Pure Left Null Space Noise

In [ ]:
# get left null space vector with squared magnitude of 4:
e_LNS = U[:, R:] @ (np.array([[1], [-1]]) * np.sqrt(2)) 
# get noisy data with noise PURELY from left null space:
y1 = y_truth + e_LNS
norm(e_LNS)**2, y1

In [ ]:
P_CS @ y_truth, P_LNS @ y_truth  # pure CS, hence LNS-projection yields zero vector

In [ ]:
P_CS @ y1, P_LNS @ y1  # y is CS+LNS, hence projections split y into CS and LNS parts

In [ ]:
# further checks on CS and LNS
np.allclose(y_truth, P_CS @ y1), np.allclose(e_LNS, P_LNS @ y1)

In [ ]:
theta_hat1 = X_li @ y1  # model training/fitting, we get an estimate for the theta vector
theta_hat1, theta_truth  # model parameters: estimated == ground truth
# estimated == ground truth
# should not be surprising for this case 1, because the noise in data y is purely in left null space
# hence the shortest projection of y towards column space yields y_truth and therefore
# theta_hat1 = theta_truth 

In [ ]:
# make a prediction with the trained model
y_hat1 = X @ theta_hat1

In [ ]:
# projection matrix P_CS
np.allclose(P_CS, X @ X_li)
# is often called hat matrix in statistics
# as the projection to column space is often
# denoted with y_hat
P_CS @ y1, y_hat1

In [ ]:
# projection matrix P_LNS
np.allclose(P_LNS, np.eye(M) - P_CS)
# extracts the residual e_LNS from the noisy data y:
P_LNS @ y1, e_LNS  # for case 1 this is precisely e_LNS

In [ ]:
# check that e_LNS lives purely in left null space
X.T @ e_LNS

In [ ]:
# e_LNS is the shortest possible left null space vector to orthogonally project y to colum space
norm(e_LNS)  # we intentionally designed to length 2 above
# hence squared magnitude is 4
norm(e_LNS)**2, e_LNS.T @ e_LNS
# this is where the concept got its name "least squares error" from:
# we actually solved the optimisation problem 
# min (||y - X theta||^2_2) w.r.t. theta ->
# min (||e_LNS||^2_2) w.r.t. theta
# by linear algebra fundamentals

In [ ]:
plt.plot(X[:,1], y_truth, 'C0o:', ms=10, label='ground truth data')
plt.plot(X[:,1], y1, 'C1d:', label='noisy data (noise from LNS only)')
plt.plot(X[:,1], y_hat1, 'C3s:', label='model prediction == ground truth')
for i in range(M):
    plt.plot([X[i,1], X[i,1]], [y_hat1[i], y1[i]], 'k-', label='e(x'+str(i)+')')
plt.xticks(np.arange(M))
plt.yticks(np.arange(10))
plt.xlabel('univariate data in: x')
plt.ylabel('univariate data out: y')
plt.title(r'Least Squares Error Solution: e(x0)$^2$+e(x1)$^2$+e(x2)$^2$+e(x3)$^2$ is min')
plt.legend()
plt.grid(True)

## Case 2: Noise from Column Space and Left Null Space

In [ ]:
# get column space + left null space vector again with squared magnitude of 4:
np.random.seed(3)
noise = U @ np.random.randn(M, 1)
noise = noise / norm(noise) * 2
#array([[-0.64094376],
#       [ 0.68438712],
#       [-0.35766768],
#       [-1.72999399]])
noise, norm(noise)**2, norm(P_CS @ noise)**2, norm(P_LNS @ noise)**2
# check that power was split roughly equally to CS and LNS

In [ ]:
# get noisy data with noise from column space AND left null space:
y2 = y_truth + noise

In [ ]:
  # model training/fitting, we get an estimate for the theta vector
theta_hat2 = X_li @ y2
theta_hat2, theta_truth  # for case 2 estimated model parameters
# are not equal to ground truth model parameters

In [ ]:
# make a prediction with the trained model
y_hat2 = X @ theta_hat2

In [ ]:
plt.plot(X[:,1], y_truth, 'C0o:', ms=2, lw=0.75, label='ground truth data')
plt.plot(X[:,1], y2, 'C1d:', label='noisy data (noise from CS + LNS)')
plt.plot(X[:,1], y_hat2, 'C3s:', label=r'model prediction $\neq$ ground truth')
for i in range(M):
    plt.plot([X[i,1], X[i,1]], [y_hat2[i], y2[i]], 'k-', label='e(x'+str(i)+')')
plt.xticks(np.arange(M))
plt.yticks(np.arange(10))
plt.xlabel('univariate data in: x')
plt.ylabel('univariate data out: y')
plt.title(r'Least Squares Error Solution: e(x0)$^2$+e(x1)$^2$+e(x2)$^2$+e(x3)$^2$ is min')
plt.legend()
plt.grid(True)

In [ ]:
P_CS @ y2  # model prediction lives purely in CS

In [ ]:
e_LNS2 = P_LNS @ y2  # residual, i.e. what the model cannot explain from the noisy!!! data
e_LNS2, norm(e_LNS2)**2
# note: the truth data is actually unknown and never accessible in practical applications
# so the actual question is, how to design and train a model such that
# the model prediction P_CS @ y2 is very close to the unknown y_truth
# this is solved by collecting lots of data and setting up an optimisation problem
# one of the simplest optimisation schemes is the least squares error optimisation

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- feel free to use the notebooks for your own purposes
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.